# 11 — B versus C paired inference

This notebook completes Reviewer 1, Minor Comment 4 by directly
comparing unconditional four-pass revision (B) with pretrained-verifier
gating (C). It uses the same frozen study cohort, patient/source
clusters, 10,000-replicate protocol, and primary metrics as Notebook 08.
It also recomputes Holm adjustment after expanding each within-stratum
family from four to five strategy contrasts.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [
    entry for entry in sys.path
    if Path(entry or ".").resolve() != implementation_dir
]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
POST_ROOT = PATHS["root"] / "post_rerun"
POST_ROOT.mkdir(parents=True, exist_ok=True)
print("Code:", RERUN_DIR)
print("Output:", POST_ROOT)

In [ ]:
import importlib
from datetime import datetime, timezone
import numpy as np, pandas as pd
from rerun_code.common import write_json
from rerun_code.config import sha256_path
import rerun_code.post_rerun as post_module
post_module = importlib.reload(post_module)
if int(getattr(post_module, "POST_RERUN_API_VERSION", 0)) < 1:
    raise ImportError("Copy the updated rerun_code/post_rerun.py and restart the kernel.")
from rerun_code.post_rerun import (
    PRIMARY_METRICS, load_statistical_frame, paired_cluster_bootstrap_difference,
    verified_per_study_path,
)
from rerun_code.statistics import paired_cluster_permutation, holm_adjust

OUTPUT = POST_ROOT / "b_vs_c"
OUTPUT.mkdir(parents=True, exist_ok=True)
per_study_path, upstream_audit = verified_per_study_path(PATHS)
frame = load_statistical_frame(per_study_path)
reps = int(CONFIG["statistics"]["permutation_replicates"])
boot_reps = int(CONFIG["statistics"]["bootstrap_replicates"])
confidence = float(CONFIG["statistics"]["confidence_level"])
seed = int(CONFIG["statistics"]["seed"])
print("Verified rows:", len(frame), "groups:", frame.groupby(["model_key", "bundle", "source_dataset"]).ngroups)

In [ ]:
parts = []
grouped = list(frame.groupby(["model_key", "bundle", "source_dataset"], dropna=False))
for index, (keys, group) in enumerate(grouped, start=1):
    arm_b = group[group["condition"] == "B_unconditional_4pass"]
    arm_c = group[group["condition"] == "C_pretrained_gate"]
    tests = paired_cluster_permutation(
        arm_b, arm_c, id_column="query_record_id", metrics=PRIMARY_METRICS,
        replicates=reps, seed=seed,
    )
    intervals = paired_cluster_bootstrap_difference(
        arm_b, arm_c, metrics=PRIMARY_METRICS, replicates=boot_reps,
        confidence_level=confidence, seed=seed,
    )
    result = tests.merge(intervals, on="metric", validate="one_to_one")
    if not np.allclose(
        result["difference_b_minus_a"], result["difference_c_minus_b"], equal_nan=True
    ):
        raise AssertionError("B-C point estimates disagree between inference engines")
    result["arm_a_condition"] = "B_unconditional_4pass"
    result["arm_b_condition"] = "C_pretrained_gate"
    result["model_key"], result["bundle"], result["source_dataset"] = keys
    parts.append(result)
    print(f"Completed stratum {index}/{len(grouped)}: {keys}")

bc = pd.concat(parts, ignore_index=True)
existing_path = PATHS["statistics"] / "paired_cluster_permutation_tests.csv"
existing = pd.read_csv(existing_path)
all_five = pd.concat([
    existing.drop(columns=["p_holm_within_strategy_family"], errors="ignore"),
    bc.drop(columns=[
        "difference_c_minus_b", "difference_ci_low", "difference_ci_high",
        "valid_bootstrap_replicates", "n_clusters",
    ]),
], ignore_index=True)
family = ["model_key", "metric", "bundle", "source_dataset"]
all_five["p_holm_five_contrast_family"] = all_five.groupby(
    family, dropna=False
)["p_value"].transform(lambda values: holm_adjust(values.tolist()))
bc = bc.merge(
    all_five[[
        "model_key", "bundle", "source_dataset", "metric", "arm_a_condition",
        "arm_b_condition", "p_holm_five_contrast_family",
    ]],
    on=["model_key", "bundle", "source_dataset", "metric", "arm_a_condition", "arm_b_condition"],
    validate="one_to_one",
)
lower_is_better = {"fer_abnormal", "omission"}
bc["holm_significant_0_05"] = bc["p_holm_five_contrast_family"] < 0.05
bc["direction_c_vs_b"] = np.where(
    bc["metric"].isin(lower_is_better),
    np.where(bc["difference_c_minus_b"] < 0, "C better", np.where(bc["difference_c_minus_b"] > 0, "C worse", "tie")),
    np.where(bc["difference_c_minus_b"] > 0, "C better", np.where(bc["difference_c_minus_b"] < 0, "C worse", "tie")),
)
bc_path = OUTPUT / "b_vs_c_paired_cluster_results.csv"
all_path = OUTPUT / "within_model_tests_five_contrasts_recomputed_holm.csv"
bc.to_csv(bc_path, index=False)
all_five.to_csv(all_path, index=False)

In [ ]:
summary_rows = []
for metric, subset in bc.groupby("metric", sort=False):
    significant = subset[subset["holm_significant_0_05"]]
    summary_rows.append({
        "metric": metric,
        "n_strata": len(subset),
        "n_holm_significant": len(significant),
        "n_significant_c_better": int((significant["direction_c_vs_b"] == "C better").sum()),
        "n_significant_c_worse": int((significant["direction_c_vs_b"] == "C worse").sum()),
        "median_difference_c_minus_b": subset["difference_c_minus_b"].median(),
        "minimum_difference_c_minus_b": subset["difference_c_minus_b"].min(),
        "maximum_difference_c_minus_b": subset["difference_c_minus_b"].max(),
    })
summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT / "b_vs_c_summary.csv"
summary.to_csv(summary_path, index=False)
status = {
    "ready": True,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    **upstream_audit,
    "n_rows": len(frame),
    "n_strata": int(frame.groupby(["model_key", "bundle", "source_dataset"]).ngroups),
    "n_b_vs_c_tests": len(bc),
    "n_holm_significant": int(bc["holm_significant_0_05"].sum()),
    "bootstrap_replicates": boot_reps,
    "permutation_replicates": reps,
    "confidence_level": confidence,
    "seed": seed,
    "multiplicity_family": "five contrasts within model-metric-bundle-source",
    "outputs": {
        str(path): sha256_path(path) for path in (bc_path, all_path, summary_path)
    },
}
write_json(OUTPUT / "notebook11_status.json", status)
display(summary)
display(bc[bc["holm_significant_0_05"]])
print(json.dumps(status, indent=2))